# appendix.ipynb

Layer-wise probe accuracy figures across three random seeds for both open models.

Set MODEL_FILES to the saved probe-result CSV files, then run the plotting cell.

Use the setup steps in the repository README before running this notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os

# =========================================================
# CONFIG
# =========================================================

MODEL_FILES = {

    "QWEN": {
        "all_layers": "/content/drive/MyDrive/probe/probe_representations_2/probe_results_all_layers.csv",
        "seed_1": "/content/drive/MyDrive/probe/probe_representations_2/probe_results_seed_1.csv",
        "seed_2": "/content/drive/MyDrive/probe/probe_representations_2/probe_results_seed_2.csv",
    },

    "FLAMINGO": {
        "all_layers": "/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_results_all_layers.csv",
        "seed_1": "/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_results_seed_1.csv",
        "seed_2": "/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_results_seed_2.csv",
    }
}

CHANCE_LEVEL = 1 / 7

# =========================================================
# HELPERS
# =========================================================

def natural_sort_key(layer):

    nums = re.findall(r"\d+", str(layer))

    if nums:
        return (
            str(layer).split(nums[-1])[0],
            int(nums[-1])
        )

    return (str(layer), 0)

def load_seed_csv(path, seed_name):

    df = pd.read_csv(path)

    # =====================================================
    # AUTO DETECT ACC COLUMN
    # =====================================================

    possible_acc_cols = [
        "test_acc",
        "accuracy",
        "acc",
        "probe_acc"
    ]

    acc_col = None

    for c in possible_acc_cols:
        if c in df.columns:
            acc_col = c
            break

    if acc_col is None:
        raise ValueError(
            f"No accuracy column found in {path}"
        )

    # =====================================================
    # AUTO DETECT LAYER COLUMN
    # =====================================================

    possible_layer_cols = [
        "layer",
        "layer_name"
    ]

    layer_col = None

    for c in possible_layer_cols:
        if c in df.columns:
            layer_col = c
            break

    if layer_col is None:
        raise ValueError(
            f"No layer column found in {path}"
        )

    # =====================================================
    # CLEAN
    # =====================================================

    df = df[[layer_col, acc_col]].copy()

    df.columns = [
        "layer",
        "accuracy"
    ]

    df["seed"] = seed_name

    return df

# =========================================================
# LOAD ALL RESULTS
# =========================================================

all_model_results = {}

for model_name, paths in MODEL_FILES.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    seed_dfs = []

    for seed_name, path in paths.items():

        print(f"Loading: {path}")

        df = load_seed_csv(
            path,
            seed_name
        )

        seed_dfs.append(df)

    merged_df = pd.concat(
        seed_dfs,
        ignore_index=True
    )

    # =====================================================
    # SORT
    # =====================================================

    merged_df["sort_key"] = merged_df[
        "layer"
    ].apply(natural_sort_key)

    merged_df = merged_df.sort_values(
        "sort_key"
    )

    all_model_results[
        model_name
    ] = merged_df

# =========================================================
# PLOT
# =========================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(22, 12)
)

axes = axes.flatten()

for plot_idx, (
    model_name,
    df
) in enumerate(all_model_results.items()):

    # =====================================================
    # SPLIT AUDIO / LLM
    # =====================================================

    audio_df = df[
        ~df["layer"].str.startswith("lm_")
    ].copy()

    llm_df = df[
        df["layer"].str.startswith("lm_")
    ].copy()

    groups = [
        ("Audio + Projector", audio_df),
        ("LLM", llm_df)
    ]

    for group_idx, (
        group_name,
        group_df
    ) in enumerate(groups):

        ax = axes[
            plot_idx * 2 + group_idx
        ]

        # =================================================
        # AGGREGATE OVER SEEDS
        # =================================================

        stats = group_df.groupby(
            "layer"
        )["accuracy"].agg(
            ["mean", "std"]
        ).reset_index()

        stats["sort_key"] = stats[
            "layer"
        ].apply(natural_sort_key)

        stats = stats.sort_values(
            "sort_key"
        )

        # =================================================
        # X
        # =================================================

        x = np.arange(len(stats))

        mean = stats["mean"].values
        std = stats["std"].fillna(0).values

        # =================================================
        # PLOT
        # =================================================

        ax.plot(
            x,
            mean,
            linewidth=2
        )

        ax.fill_between(
            x,
            mean - std,
            mean + std,
            alpha=0.20
        )

        # =================================================
        # CHANCE LINE
        # =================================================

        ax.axhline(
            CHANCE_LEVEL,
            linestyle="--",
            color="red",
            linewidth=1.5,
            label="Chance"
        )

        # =================================================
        # STYLE
        # =================================================

        ax.set_title(
            f"{model_name}\n"
            f"{group_name}\n"
            f"Probe Accuracy Across Layers"
        )

        ax.set_xlabel("Layer")
        ax.set_ylabel("Probe Accuracy")

        ax.grid(True, alpha=0.3)

        ax.legend()

# =========================================================
# FINAL
# =========================================================

plt.tight_layout()
plt.show()

# =========================================================
# SUMMARY TABLE
# =========================================================

summary_rows = []

for model_name, df in all_model_results.items():

    for group_name, group_df in [

        (
            "Audio + Projector",
            df[
                ~df["layer"].str.startswith("lm_")
            ]
        ),

        (
            "LLM",
            df[
                df["layer"].str.startswith("lm_")
            ]
        )
    ]:

        stats = group_df.groupby(
            "layer"
        )["accuracy"].agg(
            ["mean", "std"]
        ).reset_index()

        peak_row = stats.loc[
            stats["mean"].idxmax()
        ]

        summary_rows.append({

            "model": model_name,

            "group": group_name,

            "peak_layer": peak_row["layer"],

            "peak_accuracy": round(
                peak_row["mean"],
                4
            ),

            "std_at_peak": round(
                peak_row["std"],
                4
            )
        })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 60)
print("PROBE SUMMARY")
print("=" * 60)

print(summary_df)

# =========================================================
# SAVE
# =========================================================

summary_df.to_csv(
    "probe_summary.csv",
    index=False
)

print("\nSaved: probe_summary.csv")